# Fama and French Factor Model: Understanding the Factors #

In [2]:
# Import Libraries

# Data Management
import pandas as pd

# Plots
import matplotlib.pyplot as plt

# Handle Files
import sys
import os

# Import Local Functions
sys.path.append(os.path.abspath("../source"))
from config import get_tickers
from data_downloader import get_market_data
from portfolios_helper import calculate_analytics

On their website (https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html), Eugene Fama and Kenneth French share the premiums of the factor model they developed several years ago. While you can simply download them, we want to show you how to approximate those premiums using free data. :)

In [3]:
# Check the Data from Fama and French
ff_premiums = pd.read_csv(r'../additional_data/ff_size_n_value.csv')
ff_premiums.set_index('date', inplace=True)
ff_premiums.index = pd.to_datetime(ff_premiums.index)
ff_premiums.columns = ['market', 'size', 'value']

ff_premiums

In [4]:
# Create Plot
plt.figure(figsize=(10, 6))
plt.plot(ff_premiums.cumsum(), label=ff_premiums.columns, alpha=1)

# Config
plt.title('Cumulative Premiums Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns')
plt.legend()
plt.grid()

# Show
plt.show()

### Approximate Size and Value using ETFs ###

In [5]:
# Vanguard Created 9 Portfolio Categorizing by Size and Value
categories_dict = {
    "VTV": "largecap_value",
    "VOE": "midcap_value",
    "VBR": "smallcap_value",
    "VV": "largecap_blend",
    "VO": "midcap_blend",
    "VB": "smallcap_blend",
    "VUG": "largecap_growth",
    "VOT": "midcap_growth",
    "VBK": "smallcap_growth"
}

# ":)"

In [6]:
# Tickers
tickers = get_tickers(mod="5.1")

tickers

In [7]:
# Import data
etfs_returns = pd.DataFrame()

for ticker in tickers:
    df = get_market_data(
        ticker=ticker, 
        start_date='2007-01-01', 
        end_date='2025-01-01', 
        returns=True
    )
    
    returns = df['returns'].rename(ticker)
    
    etfs_returns = pd.concat([etfs_returns, returns], axis=1)

etfs_returns = etfs_returns.rename(columns=categories_dict)

In [8]:
etfs_returns

In [9]:
# Create Plot
plt.figure(figsize=(10, 6))
plt.plot(etfs_returns.cumsum(), label=etfs_returns.columns, alpha=1)

# Config
plt.title('Cumulative ETF Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns')
plt.legend()
plt.grid()

# Show
plt.show()

In [10]:
# Analytics Table
analytics_table = calculate_analytics(etfs_returns)

analytics_table.sort_values(by = 'Sharpe Ratio', ascending = False)

This does not say too much. Let us try to group the ETFs

In [11]:
# Size DataFrame
size_df = pd.DataFrame(index = etfs_returns.index)
size_df['large'] = etfs_returns[['largecap_value', 'largecap_blend', 'largecap_growth']].mean(axis=1)
size_df['mid'] = etfs_returns[['midcap_value', 'midcap_blend', 'midcap_growth']].mean(axis=1)
size_df['small'] = etfs_returns[['smallcap_value', 'smallcap_blend', 'smallcap_growth']].mean(axis=1)

In [12]:
# Value DataFrame
value_df = pd.DataFrame(index = etfs_returns.index)
value_df['value'] = etfs_returns[['largecap_value', 'midcap_value', 'smallcap_value']].mean(axis=1)
value_df['blend'] = etfs_returns[['largecap_blend', 'midcap_blend', 'smallcap_blend']].mean(axis=1)
value_df['growth'] = etfs_returns[['largecap_growth', 'midcap_growth', 'smallcap_growth']].mean(axis=1)

In [13]:
# Analytics Table
size_analytics_table = calculate_analytics(size_df)

size_analytics_table.sort_values(by = 'Sharpe Ratio', ascending = False)

In [14]:
# Analytics Table
value_analytics_table = calculate_analytics(value_df)

value_analytics_table.sort_values(by = 'Sharpe Ratio', ascending = False)

### Calculate the Premiums ###

In [15]:
# Calculate the approximation of the SMB prime
SMB = 1/3*(etfs_returns['smallcap_value'] + etfs_returns['smallcap_blend'] + etfs_returns['smallcap_growth']) - 1/3*(etfs_returns['largecap_value'] + etfs_returns['largecap_blend'] + etfs_returns['largecap_growth'])

In [16]:
# Calculate the approximation of the HML prime
HML = 1/2*(etfs_returns['largecap_value'] + etfs_returns['smallcap_value']) - 1/2*(etfs_returns['largecap_growth'] + etfs_returns['smallcap_growth'])

In [17]:
# Create Plot
plt.figure(figsize=(10, 6))
plt.plot(SMB.cumsum(), label='SMB with ETFs', alpha=1)
plt.plot(ff_premiums['size'].loc['2007':].cumsum(), label='SMB by FF', alpha=1)

# Config
plt.title('Cumulative SMB Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns')
plt.legend()
plt.grid()

# Show
plt.show()

In [18]:
# Create Plot
plt.figure(figsize=(10, 6))
plt.plot(HML.cumsum(), label='HML with ETFs', alpha=1)
plt.plot(ff_premiums['value'].loc['2007':].cumsum(), label='HML by FF', alpha=1)

# Config
plt.title('Cumulative HML Returns Time Series')
plt.xlabel('Time')
plt.ylabel('Returns')
plt.legend()
plt.grid()

# Show
plt.show()

Naturally, the differences between both sets of premiums can be explained by the universe used in their construction, but that is something we have to tolerate. If we don’t have access to market capitalization and fundamental data, we won’t be able to develop a factor model. Fortunately, there is a way to approximate these premiums using alternative data.